# Hyperparameter tuning evaluation

This notebook compares two runs on the same 15-item tuning set:

- **direct**: no reranking_papers
- **reranking_papers**: reranking_papers enabled

It loads the JSON result files, cleans old failed duplicate records, flattens equipment outputs, compares predicted equipment names against the benchmark `expected_equipment` labels, and exports CSV tables.


In [95]:
from __future__ import annotations

import json
import math
import re
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any

import pandas as pd


In [96]:
# Update these paths if your filenames differ.
RESULT_FILES = {
    "direct": Path("../data/processed/hyperparameter_tuning_15_results_no_paper_reranking_12papers.json"),
    "reranking_papers": Path("../data/processed/hyperparameter_tuning_15_results_paper_reranked_10papers.json"),
}

OUTPUT_DIR = Path("data/reranking_papers")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for mode, path in RESULT_FILES.items():
    print(mode, path, "exists:", path.exists())


direct ../data/processed/hyperparameter_tuning_15_results_no_paper_reranking_12papers.json exists: True
reranking_papers ../data/processed/hyperparameter_tuning_15_results_paper_reranked_10papers.json exists: True


In [97]:
def load_json(path: Path) -> list[dict[str, Any]]:
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError(f"Expected a JSON list in {path}, got {type(data)}")
    return data


def record_time(record: dict[str, Any]) -> str:
    """Return the best available timestamp for sorting records."""
    return (
        record.get("completed_at_utc")
        or record.get("failed_at_utc")
        or record.get("started_at_utc")
        or record.get("run_timestamp_utc")
        or ""
    )


def record_key(record: dict[str, Any]) -> str:
    """Stable key for grouping repeated attempts of the same input item."""
    return (
        record.get("input_run_id")
        or record.get("input_query_id")
        or record.get("original_query")
        or json.dumps(record, sort_keys=True, default=str)[:500]
    )


def keep_latest_prefer_completed(records: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """
    If the output JSON contains old failures plus later successes, keep one record per input.
    Completed records are preferred over failed records. Within each status, keep the latest.
    """
    grouped: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for record in records:
        grouped[record_key(record)].append(record)

    cleaned = []
    for _, group in grouped.items():
        completed = [r for r in group if r.get("run_status") == "completed"]
        candidates = completed if completed else group
        chosen = sorted(candidates, key=record_time)[-1]
        cleaned.append(chosen)

    return sorted(cleaned, key=lambda r: (r.get("input_query_id") or "", record_time(r)))


results_by_mode = {}
for mode, path in RESULT_FILES.items():
    raw_records = load_json(path)
    cleaned_records = keep_latest_prefer_completed(raw_records)
    results_by_mode[mode] = cleaned_records

    print(f"\n{mode}")
    print("raw records:", len(raw_records))
    print("cleaned unique records:", len(cleaned_records))
    print("status counts:", Counter(r.get("run_status") for r in cleaned_records))
    print("query type counts:", Counter((r.get("input_raw_benchmark_item") or {}).get("query_type") for r in cleaned_records))



direct
raw records: 15
cleaned unique records: 15
status counts: Counter({'completed': 15})
query type counts: Counter({'broad_diagnostic': 5, 'method_selection': 5, 'specific': 5})

reranking_papers
raw records: 15
cleaned unique records: 15
status counts: Counter({'completed': 15})
query type counts: Counter({'broad_diagnostic': 5, 'method_selection': 5, 'specific': 5})


In [98]:
def normalize_text(value: str | None) -> str:
    if not value:
        return ""
    value = str(value).lower()
    value = value.replace("µ", "μ")
    value = value.replace("-", " ")
    value = value.replace("–", " ")
    value = value.replace("—", " ")
    value = value.replace("_", " ")
    value = re.sub(r"[^a-z0-9μ]+", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value


def unique_keep_order(values: list[str]) -> list[str]:
    seen = set()
    out = []
    for value in values:
        normalized = normalize_text(value)
        if normalized and normalized not in seen:
            seen.add(normalized)
            out.append(value)
    return out


def expand_with_synonyms(terms: list[str], synonyms: dict[str, list[str]] | None) -> list[str]:
    expanded = list(terms)
    synonyms = synonyms or {}

    normalized_terms = {normalize_text(t) for t in terms}
    for key, vals in synonyms.items():
        key_norm = normalize_text(key)
        vals = vals or []

        if key_norm in normalized_terms:
            expanded.extend(vals)

        val_norms = {normalize_text(v) for v in vals}
        if normalized_terms.intersection(val_norms):
            expanded.append(key)
            expanded.extend(vals)

    return unique_keep_order(expanded)


def expected_terms_by_label(benchmark_item: dict[str, Any]) -> dict[str, list[str]]:
    expected = benchmark_item.get("expected_equipment") or {}
    synonyms = benchmark_item.get("synonyms") or {}

    labels = {
        "must_have": expected.get("must_have") or [],
        "best_match": expected.get("best_match") or [],
        "acceptable_alternatives": expected.get("acceptable_alternatives") or [],
        "supporting_but_not_ideal": expected.get("supporting_but_not_ideal") or [],
        "clearly_not_suitable": expected.get("clearly_not_suitable") or [],
    }

    return {
        label: expand_with_synonyms([str(x) for x in terms], synonyms)
        for label, terms in labels.items()
    }


MATCH_LABEL_SCORES = {
    "must_have": 3,
    "best_match": 3,
    "acceptable_alternatives": 2,
    "supporting_but_not_ideal": 1,
    "clearly_not_suitable": -1,
    "no_match": 0,
}

MATCH_LABEL_NAMES = {
    "must_have": "must_have_or_best_match",
    "best_match": "must_have_or_best_match",
    "acceptable_alternatives": "acceptable_alternative",
    "supporting_but_not_ideal": "supporting_but_not_ideal",
    "clearly_not_suitable": "clearly_not_suitable",
    "no_match": "no_match",
}


def names_match(predicted: str, expected: str) -> bool:
    """
    Conservative-ish string matcher for equipment names.

    It handles hyphen/case/punctuation differences and simple containment.
    This is useful for automatic screening, but you should manually inspect borderline cases.
    """
    p = normalize_text(predicted)
    e = normalize_text(expected)

    if not p or not e:
        return False

    if p == e:
        return True

    if len(p) >= 8 and len(e) >= 8:
        if p in e or e in p:
            return True

    return False


def prediction_names(equipment: dict[str, Any]) -> list[str]:
    names = []
    for key in ["equipment_name", "normalized_equipment_name"]:
        if equipment.get(key):
            names.append(str(equipment[key]))
    names.extend([str(x) for x in equipment.get("aliases", []) or []])
    return unique_keep_order(names)


def match_equipment_to_expected(
    equipment: dict[str, Any],
    benchmark_item: dict[str, Any],
) -> dict[str, Any]:
    expected_by_label = expected_terms_by_label(benchmark_item)
    predicted_names = prediction_names(equipment)

    label_order = [
        "must_have",
        "best_match",
        "acceptable_alternatives",
        "supporting_but_not_ideal",
        "clearly_not_suitable",
    ]

    best = {
        "match_label_raw": "no_match",
        "match_label": "no_match",
        "match_score": 0,
        "matched_prediction_name": None,
        "matched_expected_name": None,
    }

    for label in label_order:
        for predicted in predicted_names:
            for expected in expected_by_label[label]:
                if names_match(predicted, expected):
                    score = MATCH_LABEL_SCORES[label]
                    if score > best["match_score"]:
                        best = {
                            "match_label_raw": label,
                            "match_label": MATCH_LABEL_NAMES[label],
                            "match_score": score,
                            "matched_prediction_name": predicted,
                            "matched_expected_name": expected,
                        }

    return best


In [99]:
def flatten_equipment_records(results_by_mode: dict[str, list[dict[str, Any]]]) -> pd.DataFrame:
    rows = []

    for mode, records in results_by_mode.items():
        for record in records:
            benchmark_item = record.get("input_raw_benchmark_item") or {}
            query_id = record.get("input_query_id")
            query_type = benchmark_item.get("query_type")
            status = record.get("run_status")

            equipment_list = record.get("aggregated_equipment") or []

            if status != "completed":
                rows.append({
                    "mode": mode,
                    "query_id": query_id,
                    "query_type": query_type,
                    "run_status": status,
                    "rank": None,
                    "equipment_name": None,
                    "match_label": "run_failed",
                    "match_score": None,
                    "error": record.get("error"),
                })
                continue

            if not equipment_list:
                rows.append({
                    "mode": mode,
                    "query_id": query_id,
                    "query_type": query_type,
                    "run_status": status,
                    "rank": None,
                    "equipment_name": None,
                    "match_label": "no_equipment_returned",
                    "match_score": 0,
                    "error": None,
                })
                continue

            for rank, equipment in enumerate(equipment_list, start=1):
                match = match_equipment_to_expected(equipment, benchmark_item)

                rows.append({
                    "mode": mode,
                    "query_id": query_id,
                    "query_type": query_type,
                    "case_type": benchmark_item.get("case_type"),
                    "run_status": status,
                    "used_reranking_papers": record.get("used_reranking_papers"),
                    "num_subproblems_run": record.get("num_subproblems_run"),
                    "rank": rank,
                    "equipment_name": equipment.get("equipment_name"),
                    "normalized_equipment_name": equipment.get("normalized_equipment_name"),
                    "equipment_type": equipment.get("equipment_type"),
                    "best_relevance_label": equipment.get("best_relevance_label"),
                    "max_confidence_score": equipment.get("max_confidence_score"),
                    "num_supporting_subproblems": equipment.get("num_supporting_subproblems"),
                    "num_supporting_papers": equipment.get("num_supporting_papers"),
                    "aliases": " | ".join(equipment.get("aliases", []) or []),
                    "measurement_outputs": " | ".join(equipment.get("measurement_outputs", []) or []),
                    "explanations": " | ".join(equipment.get("explanations", []) or []),
                    "match_label": match["match_label"],
                    "match_label_raw": match["match_label_raw"],
                    "match_score": match["match_score"],
                    "matched_prediction_name": match["matched_prediction_name"],
                    "matched_expected_name": match["matched_expected_name"],
                    "original_query": record.get("original_query"),
                    "source_doi": record.get("input_source_doi"),
                    "source_pdf_path": record.get("input_source_pdf_path"),
                    "expected_must_have": " | ".join(expected_terms_by_label(benchmark_item).get("must_have", [])),
                    "expected_best_match": " | ".join(expected_terms_by_label(benchmark_item).get("best_match", [])),
                    "expected_acceptable": " | ".join(expected_terms_by_label(benchmark_item).get("acceptable_alternatives", [])),
                    "expected_supporting": " | ".join(expected_terms_by_label(benchmark_item).get("supporting_but_not_ideal", [])),
                    "expected_not_suitable": " | ".join(expected_terms_by_label(benchmark_item).get("clearly_not_suitable", [])),
                })

    return pd.DataFrame(rows)


equipment_df = flatten_equipment_records(results_by_mode)
equipment_df.head(10)


,mode,query_id,query_type,case_type,run_status,used_reranking_papers,num_subproblems_run,rank,equipment_name,normalized_equipment_name,...,matched_expected_name,original_query,source_doi,source_pdf_path,expected_must_have,expected_best_match,expected_acceptable,expected_supporting,expected_not_suitable,error
0,direct,PET_packaging_barrier,broad_diagnostic,literature_supported_inventory_uncertain,completed,NaN,1.0,1.0,oxygen transmission rate tester,oxygen transmission rate tester,...,NaN,We need to evaluate a biobased PET replacement...,NaN,data/interim/pdfs/Recommendations_for_replacin...,gas permeability tester | haze meter | gas tra...,oxygen permeability analyzer | carbon dioxide ...,permeation analyzer | gas transmission tester ...,DSC | polarized optical microscope,FTIR spectrometer | rheometer | HPLC,NaN
1,direct,PET_packaging_barrier,broad_diagnostic,literature_supported_inventory_uncertain,completed,NaN,1.0,2.0,water vapor transmission rate tester,water vapor transmission rate tester,...,water vapor transmission rate tester,We need to evaluate a biobased PET replacement...,NaN,data/interim/pdfs/Recommendations_for_replacin...,gas permeability tester | haze meter | gas tra...,oxygen permeability analyzer | carbon dioxide ...,permeation analyzer | gas transmission tester ...,DSC | polarized optical microscope,FTIR spectrometer | rheometer | HPLC,NaN
2,direct,PET_packaging_barrier,broad_diagnostic,literature_supported_inventory_uncertain,completed,NaN,1.0,3.0,haze meter,haze meter,...,haze meter,We need to evaluate a biobased PET replacement...,NaN,data/interim/pdfs/Recommendations_for_replacin...,gas permeability tester | haze meter | gas tra...,oxygen permeability analyzer | carbon dioxide ...,permeation analyzer | gas transmission tester ...,DSC | polarized optical microscope,FTIR spectrometer | rheometer | HPLC,NaN
3,direct,PET_packaging_barrier,broad_diagnostic,literature_supported_inventory_uncertain,completed,NaN,1.0,4.0,colorimeter,colorimeter,...,NaN,We need to evaluate a biobased PET replacement...,NaN,data/interim/pdfs/Recommendations_for_replacin...,gas permeability tester | haze meter | gas tra...,oxygen permeability analyzer | carbon dioxide ...,permeation analyzer | gas transmission tester ...,DSC | polarized optical microscope,FTIR spectrometer | rheometer | HPLC,NaN
4,direct,acoustic_particles_diagnostic,method_selection,literature_supported_inventory_uncertain,completed,NaN,1.0,1.0,surface acoustic wave (SAW) microfluidic chip,surface acoustic wave saw microfluidic chip,...,NaN,My sample contains fragile cells and I want a ...,NaN,data/interim/pdfs/A_concise_review_of_microflu...,acoustic tweezers system | ultrasonic standing...,surface acoustic wave microfluidic device | st...,acoustic separation chip | ultrasound standing...,hydrodynamic microfluidic separator | dielectr...,optical tweezers system | centrifuge | mass sp...,NaN
5,direct,acoustic_particles_diagnostic,method_selection,literature_supported_inventory_uncertain,completed,NaN,1.0,2.0,traveling surface acoustic wave (TSAW) microfl...,traveling surface acoustic wave tsaw microflui...,...,NaN,My sample contains fragile cells and I want a ...,NaN,data/interim/pdfs/A_concise_review_of_microflu...,acoustic tweezers system | ultrasonic standing...,surface acoustic wave microfluidic device | st...,acoustic separation chip | ultrasound standing...,hydrodynamic microfluidic separator | dielectr...,optical tweezers system | centrifuge | mass sp...,NaN
6,direct,acoustic_particles_diagnostic,method_selection,literature_supported_inventory_uncertain,completed,NaN,1.0,3.0,standing surface acoustic wave (SSAW) microflu...,standing surface acoustic wave ssaw microfluid...,...,NaN,My sample contains fragile cells and I want a ...,NaN,data/interim/pdfs/A_concise_review_of_microflu...,acoustic tweezers system | ultrasonic standing...,surface acoustic wave microfluidic device | st...,acoustic separation chip | ultrasound standing...,hydrodynamic microfluidic sep

In [100]:
def dcg(relevances: list[float], k: int) -> float:
    score = 0.0
    for i, rel in enumerate(relevances[:k], start=1):
        score += (2**rel - 1) / math.log2(i + 1)
    return score


def ideal_relevances(benchmark_item: dict[str, Any], k: int) -> list[int]:
    expected = expected_terms_by_label(benchmark_item)
    rels = []
    seen = set()

    for label, rel in [
        ("must_have", 3),
        ("best_match", 3),
        ("acceptable_alternatives", 2),
        ("supporting_but_not_ideal", 1),
    ]:
        for term in expected.get(label, []):
            term_norm = normalize_text(term)
            if term_norm and term_norm not in seen:
                seen.add(term_norm)
                rels.append(rel)

    if not rels:
        rels = [3]

    rels = sorted(rels, reverse=True)
    return rels[:k]


def summarize_query_metrics(results_by_mode: dict[str, list[dict[str, Any]]], k_values=(1, 3, 5)) -> pd.DataFrame:
    rows = []

    for mode, records in results_by_mode.items():
        for record in records:
            benchmark_item = record.get("input_raw_benchmark_item") or {}
            equipment_list = record.get("aggregated_equipment") or []

            row = {
                "mode": mode,
                "query_id": record.get("input_query_id"),
                "query_type": benchmark_item.get("query_type"),
                "case_type": benchmark_item.get("case_type"),
                "run_status": record.get("run_status"),
                "used_reranking_papers": record.get("used_reranking_papers"),
                "num_subproblems_run": record.get("num_subproblems_run"),
                "num_returned_equipment": len(equipment_list),
                "original_query": record.get("original_query"),
            }

            if record.get("run_status") != "completed":
                row.update({
                    "top1_equipment": None,
                    "top1_match_label": "run_failed",
                    "top1_match_score": None,
                    "best_match_score_any_rank": None,
                    "best_match_label_any_rank": "run_failed",
                    "error": record.get("error"),
                })
                rows.append(row)
                continue

            match_scores = []
            match_labels = []
            for equipment in equipment_list:
                match = match_equipment_to_expected(equipment, benchmark_item)
                match_scores.append(match["match_score"])
                match_labels.append(match["match_label"])

            top1_equipment = equipment_list[0].get("equipment_name") if equipment_list else None
            top1_score = match_scores[0] if match_scores else 0
            top1_label = match_labels[0] if match_labels else "no_equipment_returned"
            best_score = max(match_scores) if match_scores else 0
            best_label = match_labels[match_scores.index(best_score)] if match_scores else "no_equipment_returned"

            row.update({
                "top1_equipment": top1_equipment,
                "top1_match_label": top1_label,
                "top1_match_score": top1_score,
                "best_match_score_any_rank": best_score,
                "best_match_label_any_rank": best_label,
                "strict_top1_correct": top1_score >= 3,
                "relaxed_top1_correct": top1_score >= 2,
                "strict_found_anywhere": best_score >= 3,
                "relaxed_found_anywhere": best_score >= 2,
                "error": None,
            })

            for k in k_values:
                topk_scores = match_scores[:k]
                row[f"strict_top{k}_correct"] = any(score >= 3 for score in topk_scores)
                row[f"relaxed_top{k}_correct"] = any(score >= 2 for score in topk_scores)

                ideal = ideal_relevances(benchmark_item, k)
                ideal_dcg = dcg(ideal, k)
                row[f"ndcg_at_{k}"] = dcg(topk_scores, k) / ideal_dcg if ideal_dcg > 0 else 0.0

            rows.append(row)

    return pd.DataFrame(rows)


query_metrics_df = summarize_query_metrics(results_by_mode)
query_metrics_df


,mode,query_id,query_type,case_type,run_status,used_reranking_papers,num_subproblems_run,num_returned_equipment,original_query,top1_equipment,...,strict_found_anywhere,relaxed_found_anywhere,error,ndcg_at_1,strict_top3_correct,relaxed_top3_correct,ndcg_at_3,strict_top5_correct,relaxed_top5_correct,ndcg_at_5
0,direct,PET_packaging_barrier,broad_diagnostic,literature_supported_inventory_uncertain,completed,None,1,4,We need to evaluate a biobased PET replacement...,oxygen transmission rate tester,...,True,True,None,0.000000,True,True,0.530721,True,True,0.383566
1,direct,acoustic_particles_diagnostic,method_selection,literature_supported_inventory_uncertain,completed,None,1,3,My sample contains fragile cells and I want a ...,surface acoustic wave (SAW) microfluidic chip,...,False,False,None,0.000000,False,False,0.000000,False,False,0.000000
2,direct,afm-grease-microstructure,specific,paper_grounded_local_candidate,completed,None,1,1,I need to image the fresh lithium grease micro...,atomic force microscope,...,True,True,None,1.000000,True,True,0.613147,True,True,0.613147
3,direct,antifouling_01,broad_diagnostic,literature_supported_inventory_uncertain,completed,None,1,0,I’m screening a new antifouling coating and ne...,NaN,...,False,False,None,0.000000,False,False,0.000000,False,False,0.000000
4,direct,dnaJB8-hsp70-fp,specific,paper_grounded_local_candidate,completed,None,1,3,A protein biophysics lab wants to test whether...,fluorescence polarization reader,...,True,True,None,1.000000,True,True,0.469279,True,True,0.339160
5,direct,fmi_system,broad_diagnostic,literature_supported_inventory_uncertain,completed,None,1,2,I need to compare fluorescence molecular imagi...,fluorescence molecular imaging surgical system,...,False,False,None,0.000000,False,False,0.000000,False,False,0.000000
6,direct,osc_morphology_giwaxs,specific,paper_grounded_local_candidate,completed,None,1,3,I need to verify whether adding a fullerene ad...,grazing-incidence wide-angle X-ray scattering,...,True,True,None,1.000000,True,True,0.545096,True,True,0.393955
7,direct,pcm_multiple_events,method_selection,literature_supported_inventory_uncertain,completed,None,1,3,I want to test whether a crystalline chalcogen...,atom probe tomography microscope,...,True,True,None,1.000000,True,True,1.000000,True,True,0.722727
8,direct,pef_barrier_measurement,broad_diagnostic,literature_supported_inventory_uncertain,completed,None,1,3,I need to compare a new PEF film against PET b...,oxygen transmission rate analyzer,...,False,False,None,0.000000,False,False,0.000000,False,False,0.000000
9,direct,small_vesicle_nanomech,method_selection,literature_supported_inventory_uncertain,completed,None,1,2,I have submicrometer extracellular vesicles an...,Atomic Force Microscope,...,True,True,None,1.000000,True,True,0.469279,True,True,0.339160


In [101]:
def sum_counter_dicts(dicts: list[dict[str, int] | None]) -> dict[str, int]:
    counter = Counter()
    for d in dicts:
        if d:
            counter.update(d)
    return dict(counter)


def source_summary_for_record(record: dict[str, Any]) -> dict[str, Any]:
    summaries = []
    for subproblem_result in record.get("subproblem_results") or []:
        pipeline_result = subproblem_result.get("pipeline_result") or {}
        summary = pipeline_result.get("paper_retrieval_summary") or {}
        if summary:
            summaries.append(summary)

    numeric_keys = [
        "num_papers_attempted",
        "num_papers_with_chunks",
        "num_full_text_success",
        "num_fallback_used",
        "num_skipped",
        "num_pdf_used",
        "num_html_used",
        "num_ar5iv_used",
        "num_abstract_fallback_used",
        "num_tldr_metadata_fallback_used",
        "num_metadata_fallback_used",
    ]

    out = {key: sum(s.get(key, 0) or 0 for s in summaries) for key in numeric_keys}

    attempted = out["num_papers_attempted"] or 0
    out["full_text_success_fraction"] = out["num_full_text_success"] / attempted if attempted else 0.0
    out["fallback_used_fraction"] = out["num_fallback_used"] / attempted if attempted else 0.0
    out["skipped_fraction"] = out["num_skipped"] / attempted if attempted else 0.0

    out["used_source_type_counts"] = sum_counter_dicts([s.get("used_source_type_counts") for s in summaries])
    out["fallback_source_type_counts"] = sum_counter_dicts([s.get("fallback_source_type_counts") for s in summaries])
    out["full_text_source_type_counts"] = sum_counter_dicts([s.get("full_text_source_type_counts") for s in summaries])

    return out


def flatten_source_summaries(results_by_mode: dict[str, list[dict[str, Any]]]) -> pd.DataFrame:
    rows = []

    for mode, records in results_by_mode.items():
        for record in records:
            benchmark_item = record.get("input_raw_benchmark_item") or {}
            summary = source_summary_for_record(record)

            rows.append({
                "mode": mode,
                "query_id": record.get("input_query_id"),
                "query_type": benchmark_item.get("query_type"),
                "run_status": record.get("run_status"),
                "used_reranking_papers": record.get("used_reranking_papers"),
                "num_subproblems_run": record.get("num_subproblems_run"),
                **summary,
            })

    return pd.DataFrame(rows)


source_df = flatten_source_summaries(results_by_mode)
source_df.head()


,mode,query_id,query_type,run_status,used_reranking_papers,num_subproblems_run,num_papers_attempted,num_papers_with_chunks,num_full_text_success,num_fallback_used,...,num_ar5iv_used,num_abstract_fallback_used,num_tldr_metadata_fallback_used,num_metadata_fallback_used,full_text_success_fraction,fallback_used_fraction,skipped_fraction,used_source_type_counts,fallback_source_type_counts,full_text_source_type_counts
0,direct,PET_packaging_barrier,broad_diagnostic,completed,None,1,12,12,2,10,...,0,4,0,6,0.166667,0.833333,0.0,"{'metadata': 6, 'abstract': 4, 'publisher_html...","{'metadata': 6, 'abstract': 4}",{'publisher_html': 2}
1,direct,acoustic_particles_diagnostic,method_selection,completed,None,1,12,12,4,8,...,0,7,0,1,0.333333,0.666667,0.0,"{'abstract': 7, 'publisher_html': 2, 'pdf': 2,...","{'abstract': 7, 'metadata': 1}","{'publisher_html': 2, 'pdf': 2}"
2,direct,afm-grease-microstructure,specific,completed,None,1,11,11,1,10,...,0,8,0,2,0.090909,0.909091,0.0,"{'publisher_html': 1, 'metadata': 2, 'abstract...","{'metadata': 2, 'abstract': 8}",{'publisher_html': 1}
3,direct,antifouling_01,broad_diagnostic,completed,None,1,12,12,1,11,...,0,4,7,0,0.083333,0.916667,0.0,"{'abstract': 4, 'tldr_metadata': 7, 'publisher...","{'abstract': 4, 'tldr_metadata': 7}",{'publisher_html': 1}
4,direct,dnaJB8-hsp70-fp,specific,completed,None,1,8,8,2,6,...,0,5,1,0,0.250000,0.750000,0.0,"{'abstract': 5, 'pdf': 2, 'tldr_metadata': 1}","{'abstract': 5, 'tldr_metadata': 1}",{'pdf': 2}


In [102]:
def summarize_by_mode_and_type(query_metrics_df: pd.DataFrame) -> pd.DataFrame:
    completed = query_metrics_df[query_metrics_df["run_status"] == "completed"].copy()

    summary = (
        completed
        .groupby(["mode", "query_type"], dropna=False)
        .agg(
            n_queries=("query_id", "nunique"),
            avg_returned_equipment=("num_returned_equipment", "mean"),
            strict_top1_accuracy=("strict_top1_correct", "mean"),
            relaxed_top1_accuracy=("relaxed_top1_correct", "mean"),
            strict_top3_accuracy=("strict_top3_correct", "mean"),
            relaxed_top3_accuracy=("relaxed_top3_correct", "mean"),
            strict_found_anywhere=("strict_found_anywhere", "mean"),
            relaxed_found_anywhere=("relaxed_found_anywhere", "mean"),
            mean_ndcg_at_3=("ndcg_at_3", "mean"),
            mean_ndcg_at_5=("ndcg_at_5", "mean"),
        )
        .reset_index()
    )

    return summary


mode_type_summary_df = summarize_by_mode_and_type(query_metrics_df)
mode_type_summary_df


,mode,query_type,n_queries,avg_returned_equipment,strict_top1_accuracy,relaxed_top1_accuracy,strict_top3_accuracy,relaxed_top3_accuracy,strict_found_anywhere,relaxed_found_anywhere,mean_ndcg_at_3,mean_ndcg_at_5
0,direct,broad_diagnostic,5,2.4,0.2,0.2,0.4,0.4,0.4,0.4,0.200000,0.144545
1,direct,method_selection,5,2.8,0.4,0.4,0.4,0.4,0.4,0.4,0.302315,0.218491
2,direct,specific,5,2.2,0.6,0.6,0.6,0.8,0.6,0.8,0.429764,0.373512
3,reranking_papers,broad_diagnostic,5,2.2,0.4,0.4,0.6,0.6,0.6,0.6,0.353072,0.267695
4,reranking_papers,method_selection,5,2.2,0.4,0.4,0.6,0.6,0.6,0.6,0.243099,0.175694
5,reranking_papers,specific,5,1.8,0.6,0.6,0.6,0.6,0.6,0.6,0.530784,0.444602


In [103]:
overall_summary_df = (
    query_metrics_df[query_metrics_df["run_status"] == "completed"]
    .groupby("mode")
    .agg(
        n_queries=("query_id", "nunique"),
        strict_top1_accuracy=("strict_top1_correct", "mean"),
        relaxed_top1_accuracy=("relaxed_top1_correct", "mean"),
        strict_top3_accuracy=("strict_top3_correct", "mean"),
        relaxed_top3_accuracy=("relaxed_top3_correct", "mean"),
        mean_ndcg_at_3=("ndcg_at_3", "mean"),
        mean_ndcg_at_5=("ndcg_at_5", "mean"),
        avg_returned_equipment=("num_returned_equipment", "mean"),
    )
    .reset_index()
)

overall_summary_df


,mode,n_queries,strict_top1_accuracy,relaxed_top1_accuracy,strict_top3_accuracy,relaxed_top3_accuracy,mean_ndcg_at_3,mean_ndcg_at_5,avg_returned_equipment
0,direct,15,0.400000,0.400000,0.466667,0.533333,0.310693,0.245516,2.466667
1,reranking_papers,15,0.466667,0.466667,0.600000,0.600000,0.375652,0.295997,2.066667


In [104]:
source_summary_df = (
    source_df[source_df["run_status"] == "completed"]
    .groupby(["mode", "query_type"], dropna=False)
    .agg(
        n_queries=("query_id", "nunique"),
        avg_papers_attempted=("num_papers_attempted", "mean"),
        avg_full_text_success=("num_full_text_success", "mean"),
        avg_fallback_used=("num_fallback_used", "mean"),
        avg_pdf_used=("num_pdf_used", "mean"),
        avg_html_used=("num_html_used", "mean"),
        avg_abstract_fallback_used=("num_abstract_fallback_used", "mean"),
        avg_metadata_fallback_used=("num_metadata_fallback_used", "mean"),
        mean_full_text_fraction=("full_text_success_fraction", "mean"),
        mean_fallback_fraction=("fallback_used_fraction", "mean"),
    )
    .reset_index()
)

source_summary_df


,mode,query_type,n_queries,avg_papers_attempted,avg_full_text_success,avg_fallback_used,avg_pdf_used,avg_html_used,avg_abstract_fallback_used,avg_metadata_fallback_used,mean_full_text_fraction,mean_fallback_fraction
0,direct,broad_diagnostic,5,11.8,2.8,9.0,1.6,0.0,5.4,2.2,0.236364,0.763636
1,direct,method_selection,5,12.0,1.8,10.2,1.2,0.0,7.2,2.2,0.150000,0.850000
2,direct,specific,5,11.0,2.4,8.6,2.0,0.0,5.6,2.4,0.218182,0.781818
3,reranking_papers,broad_diagnostic,5,10.0,1.4,8.6,1.2,0.0,7.6,0.2,0.140000,0.860000
4,reranking_papers,method_selection,5,10.0,2.0,8.0,1.8,0.0,7.0,0.0,0.200000,0.800000
5,reranking_papers,specific,5,10.0,3.0,7.0,2.0,0.0,6.2,0.8,0.300000,0.700000


In [105]:
comparison_df = (
    query_metrics_df
    .pivot_table(
        index=["query_id", "query_type"],
        columns="mode",
        values=[
            "run_status",
            "top1_equipment",
            "top1_match_label",
            "top1_match_score",
            "strict_top1_correct",
            "relaxed_top1_correct",
            "strict_top3_correct",
            "relaxed_top3_correct",
            "ndcg_at_3",
            "num_returned_equipment",
            "num_subproblems_run",
        ],
        aggfunc="first",
    )
)

comparison_df.columns = [f"{value}_{mode}" for value, mode in comparison_df.columns]
comparison_df = comparison_df.reset_index()
comparison_df


,query_id,query_type,ndcg_at_3_direct,ndcg_at_3_reranking_papers,num_returned_equipment_direct,num_returned_equipment_reranking_papers,num_subproblems_run_direct,num_subproblems_run_reranking_papers,relaxed_top1_correct_direct,relaxed_top1_correct_reranking_papers,...,strict_top1_correct_direct,strict_top1_correct_reranking_papers,strict_top3_correct_direct,strict_top3_correct_reranking_papers,top1_equipment_direct,top1_equipment_reranking_papers,top1_match_label_direct,top1_match_label_reranking_papers,top1_match_score_direct,top1_match_score_reranking_papers
0,PET_packaging_barrier,broad_diagnostic,0.530721,0.530721,4,4,1,1,False,False,...,False,False,True,True,oxygen transmission rate tester,oxygen transmission rate tester,no_match,no_match,0,0
1,acoustic_particles_diagnostic,method_selection,0.000000,0.234639,3,5,1,1,False,False,...,False,False,False,True,surface acoustic wave (SAW) microfluidic chip,optical tweezers,no_match,no_match,0,0
2,afm-grease-microstructure,specific,0.613147,0.613147,1,1,1,1,True,True,...,True,True,True,True,atomic force microscope,atomic force microscope,must_have_or_best_match,must_have_or_best_match,3,3
3,antifouling_01,broad_diagnostic,0.000000,0.000000,0,0,1,1,False,False,...,False,False,False,False,NaN,NaN,no_equipment_returned,no_equipment_returned,0,0
4,dnaJB8-hsp70-fp,specific,0.469279,0.000000,3,2,1,1,True,False,...,True,False,True,False,fluorescence polarization reader,fluorescence polarization assay setup,must_have_or_best_match,no_match,3,0
5,fmi_system,broad_diagnostic,0.000000,0.469279,2,4,1,1,False,True,...,False,True,False,True,fluorescence molecular imaging surgical system,intraoperative near-infrared fluorescence imag...,no_match,must_have_or_best_match,0,3
6,osc_morphology_giwaxs,specific,0.545096,0.765361,3,2,1,1,True,True,...,True,True,True,True,grazing-incidence wide-angle X-ray scattering,grazing-incidence wide-angle X-ray scattering,must_have_or_best_match,must_have_or_best_match,3,3
7,pcm_multiple_events,method_selection,1.000000,0.469279,3,1,1,1,True,True,...,True,True,True,True,atom probe tomography microscope,Atom probe tomography system,must_have_or_best_match,must_have_or_best_match,3,3
8,pef_barrier_measurement,broad_diagnostic,0.000000,0.765361,3,3,1,1,False,True,...,False,True,False,True,oxygen transmission rate analyzer,oxygen transmission rate tester,no_match,must_have_or_best_match,0,3
9,small_vesicle_nanomech,method_selection,0.469279,0.469279,2,1,1,1,True,True,...,True,True,True,True,Atomic Force Microscope,Atomic force microscope,must_have_or_best_match,must_have_or_best_match,3,3


In [68]:
equipment_path = OUTPUT_DIR / "tuning_equipment_level_results.csv"
query_metrics_path = OUTPUT_DIR / "tuning_query_level_metrics.csv"
mode_type_summary_path = OUTPUT_DIR / "tuning_mode_type_summary.csv"
overall_summary_path = OUTPUT_DIR / "tuning_overall_summary.csv"
source_path = OUTPUT_DIR / "tuning_source_availability.csv"
comparison_path = OUTPUT_DIR / "tuning_direct_vs_reranking_papers_comparison.csv"

equipment_df.to_csv(equipment_path, index=False)
query_metrics_df.to_csv(query_metrics_path, index=False)
mode_type_summary_df.to_csv(mode_type_summary_path, index=False)
overall_summary_df.to_csv(overall_summary_path, index=False)
source_summary_df.to_csv(source_path, index=False)
comparison_df.to_csv(comparison_path, index=False)

print("Wrote:")
for path in [
    equipment_path,
    query_metrics_path,
    mode_type_summary_path,
    overall_summary_path,
    source_path,
    comparison_path,
]:
    print("-", path)


Wrote:
- data/reranking_papers/tuning_equipment_level_results.csv
- data/reranking_papers/tuning_query_level_metrics.csv
- data/reranking_papers/tuning_mode_type_summary.csv
- data/reranking_papers/tuning_overall_summary.csv
- data/reranking_papers/tuning_source_availability.csv
- data/reranking_papers/tuning_direct_vs_reranking_papers_comparison.csv


In [35]:
cols = [
    "mode",
    "query_id",
    "query_type",
    "top1_equipment",
    "top1_match_label",
    "top1_match_score",
    "best_match_label_any_rank",
    "best_match_score_any_rank",
    "num_returned_equipment",
]
query_metrics_df.sort_values(["query_type", "query_id", "mode"])[cols]


,mode,query_id,query_type,top1_equipment,top1_match_label,top1_match_score,best_match_label_any_rank,best_match_score_any_rank,num_returned_equipment
15,decomposition,PET_packaging_barrier,broad_diagnostic,Byk HazeGuard,no_match,0,acceptable_alternative,2,4
0,direct,PET_packaging_barrier,broad_diagnostic,Fourier Transform Infrared (FTIR) spectrometer...,no_match,0,supporting_but_not_ideal,1,2
18,decomposition,antifouling_01,broad_diagnostic,QCM-D instrument,must_have_or_best_match,3,must_have_or_best_match,3,1
3,direct,antifouling_01,broad_diagnostic,Quartz crystal microbalance with dissipation m...,must_have_or_best_match,3,must_have_or_best_match,3,2
20,decomposition,fmi_system,broad_diagnostic,fluorescence imaging system,must_have_or_best_match,3,must_have_or_best_match,3,4
5,direct,fmi_system,broad_diagnostic,fluorescence imaging system,must_have_or_best_match,3,must_have_or_best_match,3,4
23,decomposition,pef_barrier_measurement,broad_diagnostic,gas permeability / permeation measurement system,no_match,0,no_match,0,1
8,direct,pef_barrier_measurement,broad_diagnostic,oxygen transmission rate tester,no_match,0,must_have_or_best_match,3,2
26,decomposition,subcellular_lipid_map,broad_diagnostic,Secondary-ion mass spectrometer with gas-clust...,must_have_or_best_match,3,must_have_or_best_match,3,3
11,direct,subcellular_lipid_map,broad_diagnostic,time-of-flight secondary ion mass spectrometer,must_have_or_best_match,3,must_have_or_best_match,3,2


In [ ]:
equipment_df[
    (equipment_df["match_score"].fillna(0) >= 2)
][[
    "mode",
    "query_id",
    "query_type",
    "rank",
    "equipment_name",
    "match_label",
    "matched_expected_name",
    "max_confidence_score",
]]
